# Shape Alignment

Tensor operations are an integral part of neural networks and frameworks like PyTorch. But, if the shapes of a certain operation don't align, that operation will throw an error. Tensor concatenation, sum and multiplications are some common operations done when developing neural networks and that require shapes to be aligned in order for the operation to be completed correctly. Below we can see some common practices adopted by popular architectures in order to align shapes.

**Common Practices in Popular Architectures**
| **Architecture** | **Alignment Method**                     | **Operation**      |
|-------------------|------------------------------------------|--------------------|
| **ResNet**        | 1x1 conv + strided conv                  | Addition           |
| **U-Net**         | Cropping skip connection + concatenation| Concatenation      |

In this notebook we are going to explore different techniques of aligning tensors to their expected shapes.

In [42]:
# Ensures versions are correct
! pip install torch==2.3.0 numpy==1.25.2 pillow==9.4.0 torchvision==0.18

import torch
import numpy as np
import PIL

print(f"Torch version: {torch.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"PIL version: {PIL.__version__}")
print(f"GPU enabled: {torch.cuda.is_available()}")

Torch version: 2.3.0+cu121
Numpy version: 1.25.2
PIL version: 9.4.0
GPU enabled: True


## Simplistic/Instinctive Techniques

A few of the most intuitive ways of aligning shapes are to crop information, add padding or reshape a tensor. Although they seem rather drastic, as some information might not be aligned after the transformation, these techniques are used in common architectures.

### Reshape

One of the most intuitive ways of aligning shapes is to reshape a tensor.

In [ ]:
import torch
torch.manual_seed(42)
x = torch.rand((3*28,28))
y = torch.rand((3, 28, 28))
x.shape, y.shape

In [ ]:
try:
    torch.concat((x, y), dim=0)
except RuntimeError as e:
    print(f"Error: {e}")

In [ ]:
x2 = x.reshape((3, 28, 28))
x2.shape

In [ ]:
torch.concat((x2, y), dim=0).shape

But what happens if the number of elements is not the same?

In [ ]:
import torch
torch.manual_seed(42)
# +1 in the last dimension
x = torch.rand((3*28,29))
y = torch.rand((3, 28, 28))
x.shape, y.shape

In [ ]:
try:
    x2 = x.reshape((3, 28, 28))
except RuntimeError as e:
    print(f"Error: {e}")

original_x = torch.randn(3, 28, 28)
# notice the difference in the number of elements
print(x.flatten().shape)
print(original_x.flatten().shape)

as you can see, one of the downsides of using basic reshapes is that the number of elements in the original tensor and in the desired new shape must be the same.

### Crop

Another way of dealing with different shapes is to crop the tensor information. 

In [ ]:
import torch
torch.manual_seed(42)
x = torch.rand((1, 3, 30, 30))
t = torch.nn.Conv2d(3, 5, kernel_size=3, stride=1)
y = t(x)
x.shape, y.shape

In [ ]:
try:
    torch.concat((x, y), dim=1)
except RuntimeError as e:
    print(f"Error: {e}")

In [ ]:
# let's first find out by how much we need to crop the input
diffX = abs(y.shape[-1] - x.shape[-1]) # -1 in case input is CHW instead of BCHW
diffY = abs(y.shape[-2] - x.shape[-2])
diffX, diffY

In [ ]:
# Let's crop it:
crop_left = diffX // 2
crop_right = diffX - crop_left
crop_bottom = diffY // 2
crop_top = diffY - crop_bottom
cropped = x[..., crop_left:-crop_right,crop_bottom:-crop_top]
cropped.shape

In [ ]:
torch.concat((cropped, y), dim=1).shape

### Padding

Instead of cropping we can also add padding.

In [ ]:
import torch
torch.manual_seed(42)
x = torch.rand((1, 5, 28, 28))
t = torch.nn.ConvTranspose2d(5, 3, kernel_size=3, stride=1)
y = t(x)
x.shape, y.shape

In [ ]:
try:
    torch.concat((x, y), dim=1)
except RuntimeError as e:
    print(f"Error: {e}")

In [ ]:
diffX = abs(y.shape[-1] - x.shape[-1]) # -1 in case input is CHW instead of BCHW
diffY = abs(y.shape[-2] - x.shape[-2])
diffX, diffY

In [ ]:
pad_left = diffX // 2
pad_right = diffX - pad_left
pad_bottom = diffY // 2
pad_top = diffY - pad_bottom

padded = torch.nn.functional.pad(x, (pad_bottom, pad_top, pad_left, pad_right))
padded.shape

In [ ]:
torch.concat((padded, y), dim=1).shape

## Other Techniques

More complex techniques might be used. Learnable features might be useful, for example.

### 1x1 Convolutions

1x1 convolutions are usually used in order to upsample or downsample the number of channels. When the stride = 1, and because the kernel is 1x1, the spatial dimension will not be affected. Therefore, the only dimension affected is the number of channels.

One important aspect of it is that this method is learnable.

In [ ]:
import torch
torch.manual_seed(42)
x = torch.rand((1, 3, 28, 28))
t = torch.nn.Conv2d(3, 5, kernel_size=1, stride=1)
y = t(x)
x.shape, y.shape

In [ ]:
downsample = torch.nn.Conv2d(5, 10, kernel_size=1, stride=1)
y.shape, downsample(y).shape

### Pooling

Applying a predefined function inside a sliding window is another way of changing the spatial dimension of a tensor. These are pooling layers, that apply the max or mean functions over a sliding window. As this method is not learnable, it can be used to downsample tensors while decreasing the model complexity.

These can be Max Pooling (`nn.MaxPool2d` for example) or Average Pooling (`nn.AvgPool2d` for example)

In [ ]:
import torch
torch.manual_seed(42)
x = torch.rand((1, 3, 30, 30))
t = torch.nn.Conv2d(3, 5, kernel_size=3, stride=1)
y = t(x)
x.shape, y.shape

In [ ]:
downsample = torch.nn.MaxPool2d( # can be torch.nn.AvgPool2d also
    kernel_size=3,
    stride=1
) 
downsampled = downsample(x)
x.shape, downsampled.shape

In [ ]:
torch.concat([y, downsampled], 1).shape